# Step 4 — Initial Load (Day 1)

JDBC read of `public.poc_orders` (200 rows) from Supabase -> full snapshot into `bronze.poc_orders`.

In [ ]:
from pyspark.sql.functions import current_timestamp, lit

jdbc_url = "jdbc:postgresql://aws-0-ap-south-1.pooler.supabase.com:5432/postgres?sslmode=require"

connection_props = {
    "user": "postgres.isqcnhvlfnjszllicxqi",
    "password": "A1qaZ@Vfr$2#3",
    "driver": "org.postgresql.Driver",
}

In [ ]:
raw_orders_df = spark.read.jdbc(url=jdbc_url, table="public.poc_orders", properties=connection_props)

bronze_orders_df = (
    raw_orders_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("supabase_jdbc"))
    .withColumn("batch_id", lit("day1_initial_load"))
)

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
bronze_orders_df.write.format("delta").mode("overwrite").saveAsTable("bronze.poc_orders")

print("bronze.poc_orders row count:", spark.table("bronze.poc_orders").count())
display(spark.table("bronze.poc_orders"))